# H&M データ探索的分析 (EDA)

このノートブックでは、KaggleのH&Mデータセットを段階的に分析していきます。

## 目次
1. データ読み込み・基本統計
2. 商品データの分析
3. 顧客データの分析
4. 購買履歴の時系列分析
5. カテゴリ別売上分析
6. 季節性・トレンド分析
7. 顧客セグメンテーション(RFM)
8. レコメンドの方向性まとめ

In [ ]:
# 必要なライブラリと自作モジュールのimport
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_articles, load_customers, load_transactions
from src.analysis import (
    basic_stats, top_articles, sales_trend, plot_sales_trend,
    customer_segments, category_distribution, seasonal_analysis,
    price_distribution_by_category
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

## 1. データ読み込み

In [ ]:
articles = load_articles()
customers = load_customers()
# 全件は重いので5%サンプリング
transactions = load_transactions(sample_frac=0.05)

basic_stats(articles, 'Articles')
basic_stats(customers, 'Customers')
basic_stats(transactions, 'Transactions (5% sample)')

## 2. 商品データの分析

In [ ]:
# 商品グループの分布
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
articles['index_group_name'].value_counts().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Index Group (Top-level Category)')
articles['product_group_name'].value_counts().head(15).plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Product Group (Top 15)')
plt.tight_layout()
plt.show()

In [ ]:
# カラーバリエーションの分布
plt.figure(figsize=(12, 4))
articles['colour_group_name'].value_counts().head(20).plot.bar(color='gray')
plt.title('Color Distribution (Top 20)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. 顧客データの分析

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
customers['age'].dropna().plot.hist(bins=50, ax=axes[0], color='lightgreen', edgecolor='black')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
customers['club_member_status'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%', colors=['#ff9999', '#66b3ff', '#99ff99']
)
axes[1].set_title('Club Member Status')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 4. 購買履歴の時系列分析

In [ ]:
daily = sales_trend(transactions)
print(f'期間: {daily["t_dat"].min()} 〜 {daily["t_dat"].max()}')
print(f'総トランザクション数: {daily["n_transactions"].sum():,}')
plot_sales_trend(daily)

In [ ]:
# 曜日別の傾向
transactions['weekday'] = transactions['t_dat'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure(figsize=(10, 4))
transactions.groupby('weekday').size().reindex(weekday_order).plot.bar(color='teal')
plt.title('Transactions by Weekday')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. カテゴリ別売上

In [ ]:
cat_dist = category_distribution(articles, transactions)
plt.figure(figsize=(12, 5))
cat_dist.head(15).plot.bar(color='salmon')
plt.title('Top 15 Best-selling Product Groups')
plt.ylabel('Number of Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
price_distribution_by_category(articles, transactions)

## 6. 季節性分析

In [ ]:
seasonal = seasonal_analysis(transactions, articles)
# 上位カテゴリのみ表示
top_cats = seasonal.sum().sort_values(ascending=False).head(8).index
plt.figure(figsize=(12, 5))
for cat in top_cats:
    plt.plot(seasonal.index, seasonal[cat], marker='o', label=cat, linewidth=2)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.title('Monthly Sales by Category')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 7. 顧客セグメンテーション (RFM)

- **Recency**: 最終購買からの経過日数（小さいほど良い）
- **Frequency**: 購買頻度
- **Monetary**: 購買金額

In [ ]:
rfm = customer_segments(customers, transactions)
print(rfm.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rfm['recency'].plot.hist(bins=50, ax=axes[0], color='#ffa07a')
axes[0].set_title('Recency (days since last purchase)')
rfm['frequency'].plot.hist(bins=50, ax=axes[1], color='#87ceeb', range=(0, 50))
axes[1].set_title('Frequency (capped at 50)')
rfm['monetary'].plot.hist(bins=50, ax=axes[2], color='#90ee90', range=(0, 5))
axes[2].set_title('Monetary (capped at 5)')
plt.tight_layout()
plt.show()

In [ ]:
# 簡易セグメント分け
def segment_customer(row):
    if row['frequency'] >= 10 and row['recency'] <= 30:
        return '優良顧客'
    elif row['frequency'] >= 5 and row['recency'] <= 90:
        return 'リピーター'
    elif row['recency'] > 180:
        return '休眠'
    else:
        return 'カジュアル'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
plt.figure(figsize=(8, 5))
rfm['segment'].value_counts().plot.bar(color=['#2ecc71', '#3498db', '#95a5a6', '#e74c3c'])
plt.title('Customer Segments')
plt.xticks(rotation=0)
plt.ylabel('# of Customers')
plt.tight_layout()
plt.show()

## 8. レコメンドの方向性まとめ

EDAから得られた知見をもとに、3つのレコメンドエンジンを設計します:

### ① 人気商品レコメンド (新規ユーザー向け)
- **アプローチ**: 直近2週間の購買数 + 時間減衰重み
- **多様性**: カテゴリのラウンドロビンで偏りを防止
- **個別化**: 年齢層別の人気も用意

### ② あなたへのおすすめ (協調フィルタリング)
- **アプローチ**: Item-Item協調フィルタリング (コサイン類似度)
- **データ**: 直近90日のトランザクション（最低5回購入された商品）
- **特徴**: ユーザーの全履歴の類似商品スコアを集約

### ③ 関連商品 (コンテンツベース)
- **アプローチ**: 商品属性（カテゴリ・色・タイプ）のOne-Hot + コサイン類似度
- **特徴**: 重要属性ほど大きな重みづけ（商品タイプ x3, 色 x1.5）
- **強み**: コールドスタート（新商品）に強い

次のステップ: `python src/build_recommendations.py` で全レコメンドを事前計算！